In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import numpy as np
import matplotlib as mat
import matplotlib.pyplot as plt
%matplotlib inline
import matplotlib.colors as colors
import matplotlib.gridspec as gridspec
import pandas as pd
import tables
import math
from scipy.stats import mstats
import matplotlib as mpl
import matplotlib.font_manager as font_manager


In [3]:
livetime_yr = 11.687
livetime_s  = livetime_yr * 365.25 * 24 * 3600 # 11.687 year

In [4]:
def nan_to_zero(values):
    return np.array([0 if (isinstance(v, float) and math.isnan(v)) else v for v in values])

In [5]:
datasets_to_compare = ["21315","21316"]

hdf_file_path_thijs = "/data/user/tvaneede/GlobalFit/reco_processing/hdf/output/hese_iceprod_v7/merged/"
hdf_file_paths_thijs = {
    "21315" : f"{hdf_file_path_thijs}/HESE_MuonGun_MuonGun_21315.h5",
    "21316" : f"{hdf_file_path_thijs}/HESE_MuonGun_MuonGun_21316.h5",
    "21317" : f"{hdf_file_path_thijs}/HESE_MuonGun_MuonGun_21317.h5",
    "all"   : f"{hdf_file_path_thijs}/HESE_MuonGun_MuonGun.h5",
}

hdf_file_path_neha = "/data/ana/Diffuse/GlobalFit_Flavor/taupede/MuonGun/RecowithBfr/hdf_files/NoDeepCore/AllHESE/"
hdf_file_paths_neha = {
    "21315_Tracks" : f"{hdf_file_path_neha}/21315_Tracks.hdf5",
    "21315_Cascades" : f"{hdf_file_path_neha}/21315_Cascades.hdf5",
    "21315_DoubleCascades" : f"{hdf_file_path_neha}/21315_DoubleCascades.hdf5",

    "21316_Tracks" : f"{hdf_file_path_neha}/21316_Tracks.hdf5",
    "21316_Cascades" : f"{hdf_file_path_neha}/21316_Cascades.hdf5",
    "21316_DoubleCascades" : f"{hdf_file_path_neha}/21316_DoubleCascades.hdf5",
}

In [6]:
hdf_files_thijs = {}

for dataset_id, hdf_file_path in hdf_file_paths_thijs.items(): hdf_files_thijs[dataset_id] = tables.open_file( hdf_file_path ) 

hdf_files_neha = {}
for dataset_id, hdf_file_path in hdf_file_paths_neha.items(): hdf_files_neha[dataset_id] = tables.open_file( hdf_file_path ) 

In [7]:
# lets see how many events Neha found: 
dataset_ids = ["21317", "21316", "21315"]
subfolders_list = [ [f"{i:07d}-{i+999:07d}" for i in range(0, 20000, 1000)], 
                    [f"{i:07d}-{i+999:07d}" for i in range(0, 40000, 1000)],
                    [f"{i:07d}-{i+999:07d}" for i in range(0, 15000, 1000)] ]

for i, dataset_id in enumerate(dataset_ids):
    subfolders = subfolders_list[i]
    print(10*"-", dataset_id)
    total = 0
    for subfolder in subfolders:
        total_subfolder = 0
        # print(10*"-", dataset_id,subfolder)
        for channel in ["Tracks", "Cascades", "DoubleCascades"]:
            hdf_file_path = f"/data/ana/Diffuse/GlobalFit_Flavor/taupede/MuonGun/RecowithBfr/hdf_files/NoDeepCore/AllHESE/{dataset_id}_{channel}_{subfolder}.hdf5"
            hdf_file = tables.open_file(hdf_file_path)
            try:
                # print(channel, hdf_file.get_node("/I3EventHeader")[:]["Event"])
                total_subfolder += len(hdf_file.get_node("/I3EventHeader")[:]["Event"])
            except: 
                # print(channel, [])
                continue
        # print("total_subfolder", total_subfolder)
        total += total_subfolder
    print("total",total)

---------- 21317
total 0
---------- 21316
total 25
---------- 21315
total 21


Neha finds for AllHESE
- 21315: 21 events
- 21316: 25 events
- 21317: 0  events 

How many are there in my hdf5 file?

In [8]:
for dataset_id, hdf_file in hdf_files_thijs.items():    
    try:
        print(dataset_id,len(hdf_file.get_node("/I3EventHeader")[:]["Event"]))
        # print(hdf_file.get_node("/I3EventHeader")[:]["Event"])
    except:
        print(dataset_id,0)

21315 21
21316 25
21317 0
all 46


We both have one Nan weight event.

Neha has the same. Let's remove this event. NNMFit doesnt take it into account.

In [27]:
MuonWeightConv = hdf_files_thijs["all"].get_node("/MuonWeightConv").cols.value[:]
MuonWeightScaled = hdf_files_thijs["all"].get_node("/MuonWeightScaled").cols.value[:]
RecoETot = hdf_files_thijs["all"].get_node("/RecoETot").cols.value[:]
header_node = hdf_files_thijs["all"].get_node("/I3EventHeader")

run_id = header_node.cols.Run[:]
event_id = header_node.cols.Event[:]
subevent_id = header_node.cols.SubEvent[:]

df_thijs = pd.DataFrame({
    "Run": run_id,
    "Event": event_id,
    "SubEvent": subevent_id,
    "RecoETot": RecoETot,
    "MuonWeightConv": MuonWeightConv,
    "MuonWeightScaled" : MuonWeightScaled,
})

df_thijs[df_thijs["MuonWeightConv"].isna()]
df_thijs

,Run,Event,SubEvent,RecoETot,MuonWeightConv,MuonWeightScaled
0,21315,180,0,55741.303389,2.501770e-11,5.253716e-11
1,21315,393,0,111110.021393,4.812074e-13,1.010536e-12
2,21315,447,0,97982.991587,5.591751e-12,1.174268e-11
3,21315,266,0,69062.936357,1.248066e-11,2.620939e-11
4,21315,54,0,132049.988350,4.102527e-12,8.615306e-12
5,21315,301,0,75527.734632,6.657274e-12,1.398028e-11
6,21315,11,0,173462.629136,1.154278e-11,2.423983e-11
7,21315,231,0,112176.535544,1.731842e-11,3.636869e-11
8,21315,382,0,337855.122491,1.039705e-11,2.183380e-11
9,21315,65,0,105730.330868,1.145871e-11,2.406330e-11


In [28]:
# does neha also have this?
dfs_neha = []

for dataset_id, hdf_file in hdf_files_neha.items():
    
    try:
        # --- read weights ---
        muon_conv = hdf_file.get_node("/MuonWeightConv").cols.value[:]
        muon_scaled = hdf_file.get_node("/MuonWeightScaled").cols.value[:]
        RecoETot = hdf_file.get_node("/RecoETot").cols.value[:]
        
        # --- read header ---
        header_node = hdf_file.get_node("/I3EventHeader")
        run_id = header_node.cols.Run[:]
        event_id = header_node.cols.Event[:]
        subevent_id = header_node.cols.SubEvent[:]

        # --- build dataframe ---
        df_tmp = pd.DataFrame({
            "Run": run_id,
            "Event": event_id,
            "SubEvent": subevent_id,
            "RecoETot": RecoETot,
            "MuonWeightConv": muon_conv,
            "MuonWeightScaled": muon_scaled,
            "Dataset": dataset_id,   # keep origin info
        })

        dfs_neha.append(df_tmp)

    except tables.NoSuchNodeError:
        print(f"Skipping {dataset_id} (missing node)")
        continue

df_neha = pd.concat(dfs_neha, ignore_index=True)
df_neha

Skipping 21315_Cascades (missing node)
Skipping 21315_DoubleCascades (missing node)
Skipping 21316_DoubleCascades (missing node)


,Run,Event,SubEvent,RecoETot,MuonWeightConv,MuonWeightScaled,Dataset
0,21315,231,0,118324.105675,1.731842e-11,3.636869e-11,21315_Tracks
1,21315,264,0,68560.202478,6.035966e-11,1.267553e-10,21315_Tracks
2,21315,176,0,92265.873201,1.652937e-11,3.471168e-11,21315_Tracks
3,21315,266,0,69906.172309,1.248066e-11,2.620939e-11,21315_Tracks
4,21315,54,0,127510.311397,4.102527e-12,8.615306e-12,21315_Tracks
5,21315,274,0,276682.850685,5.265465e-13,1.105748e-12,21315_Tracks
6,21315,72,0,55764.294503,5.277225e-11,1.108217e-10,21315_Tracks
7,21315,52,0,112868.685192,1.212942e-10,2.547179e-10,21315_Tracks
8,21315,384,0,187568.650821,2.578912e-12,5.415715e-12,21315_Tracks
9,21315,65,0,119960.345391,1.145871e-11,2.406330e-11,21315_Tracks


In [29]:
# overlapping events
df_thijs_idx = df_thijs.set_index(["Run", "Event", "SubEvent"])
df_neha_idx  = df_neha.set_index(["Run", "Event", "SubEvent"])
thijs_not_in_neha = df_thijs_idx.loc[
    ~df_thijs_idx.index.isin(df_neha_idx.index)
]
neha_not_in_thijs = df_neha_idx.loc[
    ~df_neha_idx.index.isin(df_thijs_idx.index)
]
print(len(thijs_not_in_neha))
print(len(neha_not_in_thijs))

0
0


In [30]:
# ratio weights
print(sum(df_thijs[df_thijs["MuonWeightConv"]>0]["MuonWeightConv"])/sum(df_neha[df_neha["MuonWeightConv"]>0]["MuonWeightConv"]))

1.031159491245396


In [35]:
# overlapping events > 60 TeV
df_thijs_idx = df_thijs[df_thijs["RecoETot"] > 60e3].set_index(["Run", "Event", "SubEvent"])
df_neha_idx  = df_neha[df_neha["RecoETot"] > 60e3].set_index(["Run", "Event", "SubEvent"])
set_thijs = set(df_thijs_idx.index)
set_neha  = set(df_neha_idx.index)
only_thijs = set_thijs - set_neha
only_neha  = set_neha - set_thijs
print("only_thijs", only_thijs)
print("only_neha", only_neha)

only_thijs {(21315, 72, 0)}
only_neha {(21316, 2914, 0)}


How many events survive the 60 TeV cut?

In [14]:
# thijs
print("thijs", sum(hdf_files_thijs["all"].get_node("/RecoETot").cols.value[:] > 60e3))

# neha
print("neha")
for dataset_id, hdf_file in hdf_files_neha.items():
    try:
        print(dataset_id, sum(hdf_file.get_node("/RecoETot").cols.value[:] > 60e3))
    except: continue

thijs 20
neha
21315_Tracks 19
21316_Tracks 1
21316_Cascades 0


How many are classified as single, double and track according to FinalTopology?

In [17]:
# thijs
energy_mask = hdf_files_thijs["all"].get_node("/RecoETot").cols.value[:] > 60e3

for channel, channel_int in zip(["cascade","double","track"],[1,2,3]):
    mask = hdf_files_thijs["all"].get_node("/FinalTopology").cols.value[:] == channel_int
    print(channel, sum(energy_mask & mask))

cascade 0
double 0
track 20


How does this picture look with my new BDT selection

In [ ]:
import joblib

bdt_model_dir = "/data/user/tvaneede/GlobalFit/reco_processing/bdt/training/optimize_training/output"

### optimized, bdt new binning, optimal values after 900 steps
feature_name = "11features_plus_rloglmilli_econf_evtgen"
mcd_name = "simpletopology"
cut_length = 10
cut_bdt1 = 0.333333	
cut_bdt2 = 0.366667

flux_name = "hese"
model_name = f"mcd-{mcd_name}_flux-{flux_name}_feat-{feature_name}"

model = {
    "model1" : joblib.load(f"{bdt_model_dir}/{model_name}/train/bdt1_model.pkl"),
    "model2" : joblib.load(f"{bdt_model_dir}/{model_name}/train/bdt2_model.pkl"),
}

/data/user/tvaneede/software/py_venvs/py3-v4.4.1_reco-v1.1.0/lib/python3.12/site-packages/xgboost/core.py:377: FutureWarning: Your system has an old version of glibc (< 2.28). We will stop supporting Linux distros with glibc older than 2.28 after **May 31, 2025**. Please upgrade to a recent Linux distro (with glibc >= 2.28) to use future versions of XGBoost.
Note: You have installed the 'manylinux2014' variant of XGBoost. Certain features such as GPU algorithms or federated learning are not available. To use these features, please upgrade to a recent Linux distro with glibc 2.28+, and install the 'manylinux_2_28' variant.
  warnings.warn(


In [26]:
hdf_files_thijs["all"].get_node(f"/TauMonoDiff_rlogl")[:]["value"]

array([-3.48483477, -5.12181784, -4.7202661 , -1.71837149, -4.1547219 ,
       -2.27780002, -2.07215371, -2.98717146, -9.6801697 , -3.21155401,
       -4.8126695 , -5.24508162, -3.4189638 , -3.81614048, -2.14243477,
       -4.95145658, -4.06329806, -7.08506291, -2.93332463, -2.07573218,
       -7.06418968, -0.5917728 , -1.819419  , -0.08603526, -1.02698132,
       -5.53828318, -3.11377572, -0.58607949, -0.08875901, -0.02644328,
       -0.4147062 , -0.04752289, -3.6684451 , -0.83507994, -1.71565678,
       -1.92674396, -2.09701452, -0.48394072, -0.60833573, -0.5803488 ,
       -0.06771037, -0.93553443, -0.12180617, -0.54765301, -0.25250336,
       -1.77125889])

In [34]:
# import features
import sys
sys.path.append("/data/user/tvaneede/GlobalFit/reco_processing/bdt/training/optimize_training")
from features_list_dict import features_list_dict

features_list = [
    ['TauMonoDiff_rlogl', "value"],
    ['Taupede_Asymmetry', "value"],
    ['Taupede_Distance', "value"],
    ['Taupede1_Particles_energy', "value"],
    ['Taupede2_Particles_energy', "value"],
    ['cscdSBU_MonopodFit4_noDC_zenith', "value"],
    ['MonopodFit_iMIGRAD_PPB0_Delay_ice', "value"],
    ['CVStatistics_q_max_doms', "value"],
    ['cscdSBU_VertexRecoDist_CscdLLh', "value"],
    ['MonopodFit_iMIGRAD_PPB0', "energy"],
    ['cscdSBU_Qtot_HLC_log', "value"],
    ['TauMonoMilliDiff_rlogl', "value"],
    ['TauSPEMilliDiff_rlogl', "value"],
    ['RecoEConfinement', "value"],
    ['EventGeneratorDC_Thijs', "cascade_cascade_00001_distance"],
    ['RecoERatio_EventGeneratorDC_Max', "value"],
]

# Build feature array: shape (num_events, 13)
variables = [hdf_files_thijs["all"].get_node(f"/{feature[0]}")[:][feature[1]] for feature in features_list]    
features = np.column_stack(variables)

# Predict BDT scores
bdt_score1 = model["model1"].predict_proba(features)[:, 1]  # assuming binary classifier
bdt_score2 = model["model2"].predict_proba(features)[:, 1]  # assuming second model
reco_length = hdf_files_thijs["all"].get_node(f"/Taupede_Distance")[:]["value"]
reco_energy = hdf_files_thijs["all"].get_node(f"/RecoETot")[:]["value"]

In [ ]:
energy_mask = hdf_files_thijs["all"].get_node("/RecoETot").cols.value[:] > 60e3
print("total events", len(reco_energy), "above 60TeV", sum(energy_mask))

### cascade
mask_cascade = (bdt_score1 < cut_bdt1) | ((bdt_score2 > cut_bdt2) & (reco_length < cut_length))
print("cascade", sum(mask_cascade), "above 60TeV", sum(energy_mask & mask_cascade))

### double
mask_double = (bdt_score1 > cut_bdt1) & (bdt_score2 > cut_bdt2) & (reco_length > cut_length)
print("double", sum(mask_double), "above 60TeV", sum(energy_mask & mask_double))

### track
mask_track = (bdt_score1 > cut_bdt1) & (bdt_score2 < cut_bdt2)
print("double", sum(mask_track), "above 60TeV", sum(energy_mask & mask_track))

total events 46 above 60TeV 20
cascade 4 above 60TeV 0
double 0 above 60TeV 0
double 42 above 60TeV 20


Now I created my NNMFit frames

In [ ]:
import pandas as pd 


In [36]:
df = pd.read_parquet(
    "/data/user/tvaneede/GlobalFit/reco_processing/NNMFit/datasets/flavor_globalfit/hese/combined/IC86_pass2_SnowStorm_v2_FTP_Baseline_Combined_v7-v1/dataset_IC86_pass2_SnowStorm_v2_FTP_Baseline_Combined_v7-v1.parquet"
)
df_21315 = df.loc[21315]
df_21316 = df.loc[21316]

df_21315 = df_21315[np.log10(df_21315["reco_energy"]) > 4.778]
df_21316 = df_21316[np.log10(df_21316["reco_energy"]) > 4.778]

df_21315[["MuonWeight"]]

,,MuonWeight
11,0,2.423983e-11
52,0,2.547179e-10
54,0,8.615306e-12
65,0,2.406330e-11
67,0,NaN
72,0,1.108217e-10
95,0,1.081641e-10
144,0,1.864528e-11
176,0,3.471168e-11
190,0,1.805386e-10


In [34]:
df = pd.read_parquet(
    "/data/ana/Diffuse/GlobalFit_Flavor/NNMFit_Datasets/WithoutDeepCore/SnowStorm_v2_HESE_Baseline_Tracks/dataset_HESE_Tracks.parquet",
)
df_21315 = df.loc[21315]
df_21316 = df.loc[21316]

df_21315[["MuonWeight"]]

,,,MuonWeight
16611,11,0,2.423983e-11
64472,52,0,2.547179e-10
64701,54,0,8.615306e-12
67447,65,0,2.406330e-11
81338,67,0,1.443391e-11
109413,95,0,1.081641e-10
161996,144,0,1.864528e-11
211373,176,0,3.471168e-11
217525,190,0,1.805386e-10
279654,231,0,3.636869e-11


In [ ]:
# split
df = pd.read_parquet(
    "/data/user/tvaneede/GlobalFit/reco_processing/NNMFit/datasets/flavor_globalfit/hese/split/IC86_pass2_SnowStorm_v2_FTP_Baseline_Combined_v7-v1/mcd-simpletopology_flux-hese_feat-11features_plus_rloglmilli_econf_evtgen/bdt1_0.333333_bdt2_0.366667_length_10/dataset_IC86_pass2_SnowStorm_v2_FTP_Baseline_Combined_v7-v1_track.parquet"
)